# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Unit of analysis:** One row = one pseudonymized content item (page).
**Time window:** All metrics are aggregated over a trailing 90-day window ending at the time of export. The 90-day window is divided into the most recent 30 days (`last_30d`), the preceding 30 days (`prev_30d`), and the 30 days before that.

In [ ]:
import pandas as pd

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

# Verify unit of analysis (one row = one content_id)
grain_check = df.groupby('content_id').size()
print(f"Total rows: {len(df)}")
print(f"Unique content_ids: {len(grain_check)}")
print(f"Max rows per content_id: {grain_check.max()}")
if len(df) == len(grain_check):
    print("\nGrain verified: One row is exactly one content item.")

## 2. Fields: feature / label / context / excluded

- **Feature**: `search_volume`, `competition`, `cpc`, `content_type`, `main_intent`, `word_count`, `char_count`, `content_age_days`, `days_since_last_update`, `impressions_90d`, `clicks_90d`, `pageviews_90d`, `sessions_90d`, `users_90d`, `engaged_sessions_90d`, `scroll_events_90d`, `days_with_impressions`, `days_with_sessions`, `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`, and base 30d stats (`impressions_last_30d`, `impressions_prev_30d`, etc.). These are knowable facts about the page's history and content.
- **Label / proxy**: `trend_direction` (and the derived `is_declining_label`). This is the target to predict.
- **Context**: `content_id`, `client_id`. These are pseudonyms used for grouping and client-holdout splits, never for the model to learn from.
- **Excluded**: `trend_pct` (strongly leaks the label as it's used to derive `trend_direction`), `provider_used`, `model_used` (not model features per data dictionary). `avg_position == 0` should be treated as missing, not a numerical 0.

In [ ]:
features = [
    'search_volume', 'competition', 'cpc', 'content_type', 'main_intent',
    'word_count', 'char_count', 'content_age_days', 'days_since_last_update',
    'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
    'engaged_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions',
    'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct'
]
labels = ['trend_direction']
context = ['content_id', 'client_id']
excluded = ['trend_pct', 'provider_used', 'model_used']

print(f"Number of numeric & categorical features: {len(features)}")
print(f"Label to predict: {labels}")
print(f"Context columns: {context}")
print(f"Excluded columns: {excluded}")

## 3. Verify it with queries (grain, counts, missing values, windows)

Checking missing values and identifying pattern-based missingness. Missingness follows `content_type`.

In [ ]:
# Missing values check, grouped by content_type to see patterned gaps
missing_by_type = df.groupby('content_type')[['search_volume', 'word_count']].apply(lambda x: x.isnull().mean() * 100)
print("Missing percentage by content_type:\n")
print(missing_by_type)

# Check avg_position == 0 means missing
missing_pos_count = (df['avg_position'] == 0).sum()
print(f"\nRows with avg_position == 0: {missing_pos_count} (these are missing positions, not position zero)")

## 4. Data limits

- **Patterned Missingness**: `search_volume` and `word_count` are systematically missing based on `content_type`. A blind fillna(0) would leak the content type.
- **Zero is Not Always Zero**: `avg_position = 0` means no position data, not ranking first.
- **Rates Exceeding 100%**: `scroll_rate` and `ai_traffic_pct` can naturally exceed 100% since their numerators and denominators come from different tracking systems.
- **History Depth**: Only looking at a static trailing 90 days. We don't have deeper historical metrics in this starter dataset to see if an article regularly spikes and declines.

In [ ]:
# Verify rates exceeding 100%
scroll_gt_100 = (df['scroll_rate'] > 100).sum()
ai_gt_100 = (df['ai_traffic_pct'] > 100).sum()

print(f"Rows with scroll_rate > 100: {scroll_gt_100}")
print(f"Rows with ai_traffic_pct > 100: {ai_gt_100}")

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.